In [1]:
import pandas as pd

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from nselib import capital_market as cm

# Define dates as strings in DD-MM-YYYY format
start_date = '01-06-2021'
end_date = '05-01-2022'

# Use the correct function: price_volume_data
stk_data = cm.price_volume_data(symbol='TATACOFFEE', from_date=start_date, to_date=end_date)

print(stk_data.head())

       Symbol Series         Date  PrevClose  OpenPrice  HighPrice  LowPrice  \
0  TATACOFFEE     EQ  05-Jan-2022     217.80     218.50      218.7    214.30   
1  TATACOFFEE     EQ  04-Jan-2022     214.05     214.90      220.8    212.45   
2  TATACOFFEE     EQ  03-Jan-2022     213.15     213.95      216.9    212.30   
3  TATACOFFEE     EQ  31-Dec-2021     208.50     208.90      216.2    208.40   
4  TATACOFFEE     EQ  30-Dec-2021     211.35     211.45      211.5    207.90   

   LastPrice  ClosePrice  AveragePrice TotalTradedQuantity        Turnover₹  \
0     214.65      214.85        215.72           13,50,483  29,13,25,687.40   
1     217.40      217.80        217.41           34,77,443  75,60,37,979.15   
2     214.30      214.05        214.76           15,25,259  32,75,66,528.25   
3     213.70      213.15        213.37           30,45,699  64,98,54,073.20   
4     208.80      208.50        209.36            9,77,484  20,46,47,553.50   

  No.ofTrades  
0      12,079  
1      28,87

In [4]:
stk_data=stk_data[["OpenPrice","HighPrice","LowPrice","ClosePrice"]]

In [5]:
stk_data

,OpenPrice,HighPrice,LowPrice,ClosePrice
0,218.50,218.70,214.30,214.85
1,214.90,220.80,212.45,217.80
2,213.95,216.90,212.30,214.05
3,208.90,216.20,208.40,213.15
4,211.45,211.50,207.90,208.50
...,...,...,...,...
146,176.40,176.65,173.00,174.35
147,177.90,177.90,173.75,174.35
148,176.90,178.70,175.60,176.70
149,173.55,175.65,172.05,174.00


In [6]:
column="ClosePrice"

In [7]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (151, 4)


In [8]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [9]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

121
X_train length: (121, 4)
X_test length: (30, 4)
y_train length: (121, 4)
y_test length: (30, 4)


In [28]:
def cominbation(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
    test_obs = 28
    train =datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    from statsmodels.tsa.api import VAR
    for i in [1,2,3,4,5,6,7,8,9,10]:
        model = VAR(train)
        results = model.fit(i)
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
    x = model.select_order(maxlags=12)
    order=x.selected_orders["aic"]
    result = model.fit(order)
    #result.summary()
    lagged_Values = train.values[-order:]
    pred = result.forecast(y=lagged_Values,steps=28) 
    preds=pd.DataFrame(pred,columns=listt)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    from sklearn.metrics import mean_squared_error
    rmse= round(mean_squared_error(test,pred))
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test,pred)
    performance_metrics["Model"].append(listt)
    performance_metrics["RMSE"].append(rmse)
    performance_metrics["MaPe"].append(mape)
    performance_metrics["Lag"].append(order)
    performance_metrics["Test"].append(test_obs)
    perf=pd.DataFrame(performance_metrics)
    return perf,result,pred

In [29]:
performance_metrics={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [30]:
listt=["Close","High","Open","Low"]

In [31]:
perf,result,pred=cominbation(data1,listt)

['Close', 'High', 'Open', 'Low']
Order = 1
AIC:  -26.11355435674525
BIC:  -25.653878775641438

Order = 2
AIC:  -25.970178669519136
BIC:  -25.138373217936635

Order = 3
AIC:  -25.77313969121089
BIC:  -24.565226602672006

Order = 4
AIC:  -25.686417688175034
BIC:  -24.09834712068273

Order = 5
AIC:  -25.570381611073692
BIC:  -23.598029844504914

Order = 6
AIC:  -25.405950363476684
BIC:  -23.04511794057262

Order = 7
AIC:  -25.23964852443509
BIC:  -22.486058333328725

Order = 8
AIC:  -25.37974419859063
BIC:  -22.229039494730202

Order = 9
AIC:  -25.250601375239377
BIC:  -21.69834374048161

Order = 10
AIC:  -25.195712296601528
BIC:  -21.237379533160606



In [32]:
data1

,Open,High,Low,Close
0,0.682875,0.546667,0.732016,0.666932
1,0.632135,0.573333,0.702767,0.713831
2,0.618746,0.523810,0.700395,0.654213
3,0.547569,0.514921,0.638735,0.639905
4,0.583510,0.455238,0.630830,0.565978
...,...,...,...,...
146,0.089500,0.012698,0.079051,0.023052
147,0.110641,0.028571,0.090909,0.023052
148,0.096547,0.038730,0.120158,0.060413
149,0.049331,0.000000,0.064032,0.017488


In [33]:
print("performance summary")
perf

performance summary


,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0,8.040404e+13,1,28


In [34]:
print("\nMODEL SUMMARY:")
print(result.summary())


MODEL SUMMARY:
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Fri, 17, Jul, 2026
Time:                     17:38:26
--------------------------------------------------------------------
No. of Equations:         4.00000    BIC:                   -25.6539
Nobs:                     122.000    HQIC:                  -25.9268
Log likelihood:           920.485    FPE:                4.56150e-12
AIC:                     -26.1136    Det(Omega_mle):     3.88447e-12
--------------------------------------------------------------------
Results for equation Close
              coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------
const           -0.037377         0.010910           -3.426           0.001
L1.Close         0.102362         0.108261            0.946           0.344
L1.High         -0.184454         0.110964           -1.662     

In [36]:
print("\nFORECAST SHAPE:", pred.shape)
print("\nFIRST 5 PREDICTIONS:")
print(pd.DataFrame(pred, columns=['col1', 'col2', 'col3','col4']).head())

print("\nRMSE:", perf['RMSE'].iloc[-1])
print("MAPE:", perf['MaPe'].iloc[-1])


FORECAST SHAPE: (28, 4)

FIRST 5 PREDICTIONS:
       col1      col2      col3      col4
0  0.278055  0.231474  0.317717  0.317385
1  0.312791  0.263658  0.348657  0.348643
2  0.345861  0.292024  0.375457  0.378778
3  0.374398  0.315988  0.398604  0.405435
4  0.399078  0.336496  0.418568  0.428723

RMSE: 0
MAPE: 80404042552288.2
